<a href="https://colab.research.google.com/github/Angiehere/InteligenciaArtificial_y_RedesNeuronales_UANL_FIME/blob/main/ACTIVIDADES/PIA/AIRCANVAS%2BCNC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**PIA-Sistema mecatronico inteligente**



*  Angela Valeria Valdez Estrada
*  Yesenia Damaris Avila Cardona
*   Lesli Valeria Ibarra Beltran
*   Melissa Noemi Sanchez Ramos

In [ ]:
#Librerias
import cv2
import numpy as np
import mediapipe as mp
import math
import serial
import time
import pickle
from collections import deque

#modelo ya entrenado ;))
with open("modelo_mano.pkl", "rb") as f:
    modelo = pickle.load(f)
with open("scaler_mano.pkl", "rb") as f:
    scaler = pickle.load(f)

def pose_dibujar(lm):
    features = [[coord for point in lm for coord in (point.x, point.y, point.z)]]
    features = scaler.transform(features)
    return modelo.predict(features)[0] == 1
#config
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.8, min_tracking_confidence=0.8)
mp_draw = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0, cv2.CAP_MSMF)
#detecta la camara o no
if not cap.isOpened():
    print("Error: No se puede abrir la cámara")
    exit()
#canvas monitor
CANVAS_W, CANVAS_H = 640, 480
canvas = np.zeros((CANVAS_H, CANVAS_W, 3), dtype=np.uint8)
#trazo del aircanvas
prev_x, prev_y = 0, 0
draw_color = (0, 0, 255)
colors = [(0, 0, 255)] #rojo en rgb
#proceso para hacer el smooth en el trazo para mejor desempeño
SMOOTH = 5
buffer_x = deque(maxlen=SMOOTH)
buffer_y = deque(maxlen=SMOOTH)

all_strokes = []
current_stroke = []

BORRAR_FRAMES = 10
borrar_contador = 0
#conecte con cnc
#dim plotter
PLOTTER_W_MM = 145
PLOTTER_H_MM = 120
#puerto donde este el esp32
SERIAL_PORT = "COM12"
BAUDRATE = 115200
#puntos air canvas
def distance(lm, a, b, w, h):
    x1, y1 = int(lm[a].x * w), int(lm[a].y * h)
    x2, y2 = int(lm[b].x * w), int(lm[b].y * h)
    return math.sqrt((x2 - x1)**2 + (y2 - y1)**2)

def count_fingers(lm):
    tips = [8, 12, 16, 20]
    fingers = [lm[tip].y < lm[tip - 2].y for tip in tips]
    thumb_up = lm[4].x < lm[3].x
    return sum(fingers) + (1 if thumb_up else 0)
#trazos config
def simplify_stroke(stroke, tolerance=3):
    if len(stroke) < 3:
        return stroke
    simplified = [stroke[0]]
    for i in range(1, len(stroke) - 1):
        px, py = simplified[-1]
        cx, cy = stroke[i]
        if math.sqrt((cx - px)**2 + (cy - py)**2) >= tolerance:
            simplified.append(stroke[i])
    simplified.append(stroke[-1])
    return simplified
#conversion a gcode para cnc
def canvas_to_gcode(strokes, canvas_w, canvas_h, plotter_w, plotter_h):
    lines = ["G21", "G90", "M5", "G1 X0 Y0"]
    for stroke in strokes:
        stroke = simplify_stroke(stroke)
        if len(stroke) < 2:
            continue
        for i, (px, py) in enumerate(stroke):
            gx = round((px / canvas_w) * plotter_w, 2)
            gy = round(((canvas_h - py) / canvas_h) * plotter_h, 2)
            if i == 0:
                lines.append("M5")
                lines.append(f"G1 X{gx} Y{gy}")
                lines.append("M3")
            else:
                lines.append(f"G1 X{gx} Y{gy}")
    lines.append("M5")
    lines.append("G1 X0 Y0")
    return "\n".join(lines)
#proceso de envio e datos
def send_gcode_serial(gcode, port=SERIAL_PORT, baudrate=BAUDRATE):
    try:

        ser = serial.Serial()
        ser.port = port
        ser.baudrate = baudrate
        ser.timeout = 3
        ser.dtr = False
        ser.rts = False
        ser.open()

        print(f"Conectado a {port}, esperando ESP32...")
        time.sleep(2)

        # Limpiar basura del arranque
        ser.reset_input_buffer()
        ser.reset_output_buffer()

        total = [l for l in gcode.split("\n") if l.strip()]
        sent = 0

        for line in gcode.split("\n"):
            line = line.strip()
            if not line:
                continue
            ser.write((line + "\n").encode())
            ser.flush()
            time.sleep(0.05)
            response = ser.readline().decode(errors='ignore').strip()
            sent += 1
            print(f"[{sent}/{len(total)}] >> {line}  |  ESP32: {response}")

        ser.close()
        print("G-code enviado completamente.")

    except serial.SerialException as e:
        print(f"Error serial: {e}")
    except Exception as e:
        print(f"Error inesperado: {e}")

# resultados en terminal
while True:
    ret, frame = cap.read()
    if not ret or frame is None:
        print("No se pudo leer el frame")
        break

    frame = cv2.flip(frame, 1)
    frame_copy = frame.copy()
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)

    if result.multi_hand_landmarks:
        hand_landmarks = result.multi_hand_landmarks[0]
        lm = hand_landmarks.landmark
        h, w, _ = frame.shape

        raw_x, raw_y = int(lm[8].x * w), int(lm[8].y * h)
        buffer_x.append(raw_x)
        buffer_y.append(raw_y)
        cx = int(sum(buffer_x) / len(buffer_x))
        cy = int(sum(buffer_y) / len(buffer_y))

        finger_count = count_fingers(lm)
        dibujando = pose_dibujar(lm)

        if finger_count != 5 and dibujando:
            if prev_x == 0 and prev_y == 0:
                prev_x, prev_y = cx, cy
            cv2.line(canvas, (prev_x, prev_y), (cx, cy), draw_color, 15)
            prev_x, prev_y = cx, cy
            cv2.circle(frame_copy, (cx, cy), 10, (0, 255, 0), -1)
            cv2.putText(frame_copy, "posicion de dibujo", (10, 90),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            current_stroke.append((cx, cy))
        else:
            if current_stroke:
                all_strokes.append(current_stroke.copy())
                current_stroke.clear()
            prev_x, prev_y = 0, 0

        mp_draw.draw_landmarks(frame_copy, hand_landmarks, mp_hands.HAND_CONNECTIONS)

    else:
        if current_stroke:
            all_strokes.append(current_stroke.copy())
            current_stroke.clear()
        buffer_x.clear()
        buffer_y.clear()
        prev_x, prev_y = 0, 0
        borrar_contador = 0

    cv2.putText(frame_copy, "S=Enviar  | G=Guardar trazos | C=Limpiar canvas | Q=Salir",
                (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 0), 1)

    combined = cv2.addWeighted(frame_copy, 0.5, canvas, 0.5, 0)
    cv2.imshow("Air canvas CNC", combined)
#control mediante teclas
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == ord('c'):
        canvas = np.zeros((CANVAS_H, CANVAS_W, 3), dtype=np.uint8)
        all_strokes.clear()
        current_stroke.clear()
    elif key == ord('g'):
        if all_strokes:
            gcode = canvas_to_gcode(all_strokes, CANVAS_W, CANVAS_H, PLOTTER_W_MM, PLOTTER_H_MM)
            with open("dibujo.gcode", "w") as f:
                f.write(gcode)
            print("G-code guardado en dibujo.gcode")
        else:
            print("No hay trazos para exportar")
    elif key == ord('s'):
        if all_strokes:
            gcode = canvas_to_gcode(all_strokes, CANVAS_W, CANVAS_H, PLOTTER_W_MM, PLOTTER_H_MM)
            send_gcode_serial(gcode)
        else:
            print("No hay trazos para enviar")

cap.release()
cv2.destroyAllWindows()